In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

print("=== BAB 5: DATA CLEANSING ===")

# 1. Buka File Excel Mentah
df = pd.read_excel('../data/raw/Volve Production Data_Final.xlsx')

# 2. Proses Format Tanggal
df['DATEPRD'] = pd.to_datetime(df['DATEPRD'], format='mixed')

# 3. Standarisasi Desimal (Koma menjadi Titik)
kolom_angka = [
    'BORE_OIL_VOL', 'BORE_GAS_VOL', 'BORE_WAT_VOL', 
    'AVG_DOWNHOLE_PRESSURE', 'AVG_WELLHEAD_PRESSURE', 
    'AVG_TEMPERATURE', 'AVG_CHOKE_SIZE_P'
]
for kolom in kolom_angka:
    df[kolom] = df[kolom].astype(str).str.replace(',', '.').astype(float)

# 4. Standarisasi Penamaan Sumur
def rapikan_sumur(nama):
    nama = str(nama).upper().replace(' ', '').replace('_', '-')
    return re.sub(r'^F(\d)', r'F-\1', nama)
df['NPD_WELL_BORE_CODE'] = df['NPD_WELL_BORE_CODE'].apply(rapikan_sumur)

# Mengurutkan data (Sangat Penting agar interpolasi tidak lintas sumur)
df = df.sort_values(by=['NPD_WELL_BORE_CODE', 'DATEPRD']).reset_index(drop=True)

# 5. Konsistensi Fisik (BHP tidak boleh lebih kecil dari WHP)
idx_inkonsisten = df[df['AVG_DOWNHOLE_PRESSURE'] < df['AVG_WELLHEAD_PRESSURE']].index
df.loc[idx_inkonsisten, ['AVG_DOWNHOLE_PRESSURE', 'AVG_WELLHEAD_PRESSURE']] = np.nan

# 6. Outlier dan Nilai Anomali Sensor
df.loc[df['BORE_OIL_VOL'] < 0, 'BORE_OIL_VOL'] = np.nan
df.loc[df['AVG_DOWNHOLE_PRESSURE'] > 6000, 'AVG_DOWNHOLE_PRESSURE'] = np.nan
df.loc[df['AVG_WELLHEAD_PRESSURE'] > 3000, 'AVG_WELLHEAD_PRESSURE'] = np.nan

# 7. Interpolasi Linear untuk menambal data sensor yang dibuang
kolom_input_ml = ['BORE_OIL_VOL', 'BORE_GAS_VOL', 'BORE_WAT_VOL', 'AVG_DOWNHOLE_PRESSURE', 'AVG_WELLHEAD_PRESSURE', 'AVG_TEMPERATURE']
for col in kolom_input_ml:
    df[col] = df.groupby('NPD_WELL_BORE_CODE')[col].transform(lambda x: x.interpolate(method='linear'))
    # Jurus Sapu Jagat jika ada sisa NaN di ujung awal/akhir per sumur
    df[col] = df.groupby('NPD_WELL_BORE_CODE')[col].transform(lambda x: x.bfill().ffill())

# 8. Hapus Duplikasi
df = df.drop_duplicates(keep='first')

# 9. Split Data (Train & Impute)
df_target_kosong = df[df['AVG_CHOKE_SIZE_P'].isnull()].copy()
df_train_model = df[df['AVG_CHOKE_SIZE_P'].notnull()].copy()

# Ekspor ke CSV standar (index=False)
df_train_model.to_csv('../data/processed/volve_train_clean.csv', index=False)
df_target_kosong.to_csv('../data/processed/volve_impute_clean.csv', index=False)

print(f"Data Cleansing Selesai!")
print(f"Bahan Training ML : {len(df_train_model)} baris")
print(f"Bahan Imputasi    : {len(df_target_kosong)} baris")